In [59]:
import pandas as pd

In [60]:
import math


def generate_normal(mu, sigma):
    u1 = math.sin(math.pi * 2 * mu)
    u2 = math.cos(math.pi * 2 * sigma)

    z0 = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mu + z0 * sigma


def bernoulli(probability):
    value = (math.sin(probability) + 1) / 2
    return 1 if value < probability else 0


def poisson(lambda_):
    L = math.exp(-lambda_)
    k = 0
    p = 1
    while p > L:
        k += 1
        p *= (math.sin(k) + 1) / 2
    return k - 1

In [61]:
def simulate_population(
    years,
    initial_population,
    fertility_rate,
    mortality_rate,
    epidemic_rate,
    pandemic_rate,
):
    population = initial_population
    fertility_population = initial_population * 0.1
    history = []

    for year in range(years):

        growth_rate = generate_normal(0.02, 0.005)
        population += population * (
            (growth_rate * 1.000001)
            if year % 50 == 0
            else ((growth_rate - 0.999999) if year % 150 == 0 else growth_rate)
        )

        natural_mortality = generate_normal(0.00002, 0.000005)
        population -= population * natural_mortality

        epidemic_occurred = bernoulli(1 / 15)
        if epidemic_occurred:
            epidemic_mortality = generate_normal(0.30, 0.05)
            population -= population * epidemic_mortality

        pandemic_occurred = bernoulli(1 / 30)
        if pandemic_occurred:
            pandemic_mortality = generate_normal(0.50, 0.10)
            population -= population * pandemic_mortality

        accident_mortality = generate_normal(0.0005, 0.0001)
        population -= population * accident_mortality

        fertility_growth = generate_normal(0.02, 0.005)
        fertility_population += fertility_population * fertility_growth

        if year % 5 == 0:
            fertility_population -= fertility_population * 0.03
        history.append(
            {
                "Year": year,
                "Population": population,
                "Fertile": fertility_population,
                "Fertility": fertility_rate,
                "Growth": growth_rate,
            }
        )

    return pd.DataFrame(history)

In [62]:
initial_population = 1.5e9
years_to_simulate = 1000
fertility_rate = 0.02
mortality_rate = 0.00002
epidemic_rate = 0.1
pandemic_rate = 0.05

df = simulate_population(
    years_to_simulate,
    initial_population,
    fertility_rate,
    mortality_rate,
    epidemic_rate,
    pandemic_rate,
)

In [ ]:
df

In [ ]:
import random
import pandas as pd


# Define a class to represent an individual in the population
class Person:
    def __init__(self, age, gender):
        self.age = age
        self.gender = gender
        self.fertile = True  # All individuals are fertile initially

    def age_one_year(self):
        self.age += 1  # The person ages by one year
        # If the person is over 30, they become infertile
        if self.age > 30:
            self.fertile = False


# Define a function to simulate the population over time
def simulate_population(years, initial_population_size):
    # Initial population setup with 1:1 gender bias
    population = []
    for _ in range(initial_population_size):
        age = random.randint(18, 30)  # Random age between 18 and 30
        gender = random.choice(["Male", "Female"])  # Random gender
        population.append(Person(age, gender))

    # Store the population count over time
    population_history = []

    # Simulate each year
    for year in range(years):
        # Track current population
        current_population = len(population)

        # Every year, individuals age and those over 30 become infertile
        for person in population:
            person.age_one_year()

        # Reproduction logic: Assume each fertile female can reproduce with a fertile male
        fertile_females = [p for p in population if p.gender == "Female" and p.fertile]
        fertile_males = [p for p in population if p.gender == "Male" and p.fertile]

        # Calculate possible births (each fertile female can pair with a fertile male)
        possible_births = min(
            len(fertile_females), len(fertile_males)
        )  # 1:1 reproduction

        # Add new individuals to the population based on births
        for _ in range(possible_births):
            # Each new child starts with an age of 18
            gender = random.choice(["Male", "Female"])
            population.append(Person(age=18, gender=gender))

        # Add the population for the current year to the history
        population_history.append(current_population)

    return population_history



In [ ]:

# Set up simulation parameters
initial_population_size = 1000  # 1000 individuals in the initial population
years_to_simulate = 30  # Simulate for 30 years

# Run the simulation
population_history = simulate_population(years_to_simulate, initial_population_size)

# Create a DataFrame to display the population history
df = pd.DataFrame(
    {"Year": range(1, years_to_simulate + 1), "Population": population_history}
)

# Display the DataFrame
print(df)